# DELTA LAKE STORAGE

## Delta Lake

#### What is Delta Lake?
* Delta Lake is an open-source storage layer that brings ACID transaction, scalable metadata handling, data versioning, time travel capabilities and unified support for streaming/batch processing to data lakes.
* It is default storage layer in Databricks.
* Delta Lake runs on top of existing Data Lakes (cloud storage like AWS S3, ADLF, etc.).
<br>
##### Features:
* ACID transaction
* Schema Enforcement
* Scalable Metadata handling
* DML Operations
* Data Versoning/ Time Travel
* Transaction Log
* Streaming/batch data processing

## Understanding Traditional Data Storage Capabilities

### Data Warehouses

* Technical: Cleaned + Structured data
* Business: Operational data/ Analytical data
* Storage: Managed warehouse storage
* ACID transactions
* High Cost
* Schema-on-write
* Strong Data Modeling
* Analytical Processing/ Fast DB queries for Reporting
* Cons:
    * Less flexible
    * Expensive storage & compute
    * Data Preprocessing required
* Use Case: BI Reports/Dashboards


![image_1793511941367.png](./image_1793511941367.png "image_1793511941367.png")

### Data Lake
* Technical: Wide varity of data  
* Business: Central repository for all enterprise data
* Storage: Object Storage
* Flexible
* Low storage cost
* Schema-on-read
* Raw data is preserved (uncleaned)
* Usually built on object storage
* Very large scale  
* Use Case: Big Data, AI/ML, IoT, Analytics
* Cons:
    * Data Swamp
    * Lack of ACID Transactions

![image_1793511987586.png](./image_1793511987586.png "image_1793511987586.png")

## Delta Table Architecture

* Delta Table is essentially Data files + Transactional Log + Metadata
* When a Delta Table is created, a new directory gets created in the specified storage location (usually in ADLS Gen2, S3, or local DBFS). This directory contains:
    * _delta_log (directory): Stores the metadata + transaction logs.
    * .parquet (files): The actual data stored in columnar format.
* With each commit a delta log file is created along with its corresponding parquet file. 

![image_1793520855468.png](./image_1793520855468.png "image_1793520855468.png")

#### _delta_log

* all the transactional log files are stored in this _delta_log directory
* it tells Delta which Parquet files constitute the table at each version.
* these transacttional logs contains the metadata + transaction history
* this is what makes ACID + Time Travel possible
* this directory contains:
    * .json files: Every change to the table is recorded as a JSON file.
    * .checkpoint.parquet files: A compacted representation of the Delta transaction-log state.

### .json log files



* _**Each json represents an action**_
* Delta json log file contains the actions and information associated with a particular table commit
* Json/metadata file contains info. related to actions and associated metadata for a particular commit, corresponsing parquet files, parititons and etc.
* Since the transaction log preserves the history of each commit, it makes Time Travel possible and helps optimize reads by allowing Delta to efficiently identify the relevant data files that need to be read.
* Whenever commit happens it creates a json file inside _delta_log directory that contains information related to that partuclar commit and corresponding parquet file. 
* _json log_ : _parquet_: 1:N  
* These actions describe what changed in that commit, such as:- 
    * add → a Parquet file was added to the logical table
    * remove → a Parquet file was removed from the logical table
    * metaData → table schema, partition columns, table properties, etc.
    * protocol → Delta reader/writer protocol requirements
    * commitInfo → information about the operation/commit
    * txn → transaction/application information when applicable

In [0]:
%sql
    SELECT commitInfo 
    FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
    WHERE commitInfo IS NOT NULL;

```
"commitInfo": { 
    "timestamp": 1587968586154,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "ErrorIfExists",
      "partitionBy": []
    },
    "isBlindAppend": true
  }
  ```  

In [0]:
%sql
SELECT metaData
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE "metaData IS NOT NULL;


```
  "metaData": {
    "id": "5fba94ed-9794-4965-ba6e-6ee3c0d22af9",
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}}]}",
    "partitionColumns": [],
    "configuration": {},
    "createdTime": 1587968585495
  ```

In [0]:
%sql
SELECT add
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE add IS NOT NULL LIMIT 1;

```
{
      "path": "part-00000-a72b1fb3-f2df-41fe-a8f0-e65b746382dd-c000.snappy.parquet",
      "partitionValues": {},
      "size": 262,
      "modificationTime": 1587968586000,
      "dataChange": true
    }
```

### .checkpoint.parquet

* **Snapshot of the Delta table's state at a particular version**
* After every n commits, a checkpoint .parquet is written to avoid reading thousands of JSON files.
* Over time, a Delta table can accumulate many JSON transaction-log files. Reading and replaying a large number of JSON files to reconstruct the current table state can become increasingly expensive. Therefore, Delta periodically creates a checkpoint, which stores the accumulated table state in a compact Parquet file. This allows Delta to start from the checkpoint and process only the JSON log files created after that checkpoint, reducing the number of files and I/O operations required to reconstruct the table state.
* A checkpoint action accumulates and compacts the actions from the Delta JSON logs, storing only the resulting state needed to represent a particular table version.

```text
JSON Logs
   ↓
ADD / REMOVE / UPDATE actions
   ↓
Accumulate + resolve actions
   ↓
Checkpoint
   ↓
Resulting state
   ↓
Table state at Version N
```

In [0]:
# Disable the safety check that forces format("delta")
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")
# Force read the individual parquet file
df_spark = spark.read.format("parquet").load("/Volumes/workspace/default/demo/00000000000000000002.checkpoint.parquet")
display(df_spark)
# Reset the configuration back to true for safety
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "true")


In [0]:
# Here we can see all the 'columns' that contains all the actions after being compacted
df_spark.columns

In [0]:
df_pd = df_spark.select('add').where(df_spark.add.isNotNull()).limit(1).toPandas()

In [0]:
df_pd['add'].tolist()

Important Links:
* https://github.com/delta-io/delta-rs/tree/main/crates/test/tests/data/simple_table